# 03 — Baseline Models

Models:
1. Median baseline
2. Ridge Regression on `log(1 + price)` with one-hot encoding

These create the benchmark that advanced models must beat on the **same grouped test set**.

In [1]:
from pathlib import Path
import sys, json, os

def _find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "bonbanh_usedcar_dataset.csv").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError("Project root not found. Keep notebooks inside the CarValue_AI_Complete_Project folder.")

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /mnt/data/CarValue_AI_Complete_Project_v2


In [2]:
import time
import pandas as pd
import numpy as np
from src.carvalue_utils import (
    ensure_modeling_artifacts, get_train_test, TARGET,
    median_baseline_predictions, fit_ridge_bundle, regression_metrics, save_bundle
)

model_df, reference_year = ensure_modeling_artifacts(PROJECT_ROOT)
train, test = get_train_test(model_df)
y_test = test[TARGET].to_numpy(float)
print(train.shape, test.shape)


(18591, 31) (4595, 31)


In [3]:
# Median baseline
median_pred = median_baseline_predictions(train, test)
median_metrics = regression_metrics(y_test, median_pred)
print("Median train target:", f"{train[TARGET].median()/1e6:,.1f} million VND")
display(pd.DataFrame([median_metrics], index=["Median Baseline"]).round(4))


Median train target: 560.0 million VND


,MAE_VND,MAE_million_VND,RMSE_VND,RMSE_million_VND,R2,RMSLE
Median Baseline,6.285837e+08,628.5837,1.343907e+09,1343.9072,-0.1009,0.9065


In [4]:
# Ridge Regression
t0 = time.perf_counter()
ridge = fit_ridge_bundle(train, reference_year, alpha=10.0)
ridge_seconds = time.perf_counter() - t0
ridge_pred = ridge.predict(test)
ridge_metrics = regression_metrics(y_test, ridge_pred)
ridge_metrics["Train_seconds"] = ridge_seconds

save_bundle(ridge, PROJECT_ROOT / "models" / "ridge.joblib")
display(pd.DataFrame([ridge_metrics], index=["Ridge Regression"]).round(4))
print(f"Training time: {ridge_seconds:.2f}s")


,MAE_VND,MAE_million_VND,RMSE_VND,RMSE_million_VND,R2,RMSLE,Train_seconds
Ridge Regression,1.415643e+08,141.5643,4.566853e+08,456.6853,0.8729,0.2075,0.1389


Training time: 0.14s


In [5]:
metrics = []
metrics.append({"Model": "Median Baseline", **median_metrics, "Train_seconds": 0.0})
metrics.append({"Model": "Ridge Regression", **ridge_metrics})
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(PROJECT_ROOT / "artifacts" / "metrics_baseline.csv", index=False)

pred = pd.DataFrame({
    "row_id": test.index,
    "actual_price_vnd": y_test,
    "Median Baseline": median_pred,
    "Ridge Regression": ridge_pred,
})
pred.to_csv(PROJECT_ROOT / "artifacts" / "predictions_baseline.csv", index=False)
display(metrics_df.sort_values("MAE_VND").round(4))


,Model,MAE_VND,MAE_million_VND,RMSE_VND,RMSE_million_VND,R2,RMSLE,Train_seconds
1,Ridge Regression,1.415643e+08,141.5643,4.566853e+08,456.6853,0.8729,0.2075,0.1389
0,Median Baseline,6.285837e+08,628.5837,1.343907e+09,1343.9072,-0.1009,0.9065,0.0000
